# 🔢 Notebook 03 — NumPy MLP from First Principles
**NB6057CEM CW1 | T2 (12 marks)**

Builds a one-hidden-layer MLP entirely in NumPy (no autograd), trains it on a reduced 4-class subset, verifies gradients numerically, and demonstrates overfitting on a tiny batch.

> ⚠️ You must be able to explain every line of this notebook in the demo.

---

## Setup & Load Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json, os
from google.colab import drive

drive.mount('/content/drive')

# Restore splits and preprocessing from Drive if not found locally
os.makedirs('splits', exist_ok=True)
os.makedirs('figures', exist_ok=True)
if not os.path.exists('splits/split_indices.npz'):
    print('Syncing splits from Google Drive...')
    !cp -r /content/drive/MyDrive/ann_cw1/splits/* splits/

# Load pre-built array from Drive
data = np.load('/content/drive/MyDrive/ann_cw1/eurosat_64.npz')
X_full, y_full = data['X'], data['y']

# Load split indices
splits = np.load('splits/split_indices.npz')
train_idx = splits['train']

print(f'Full dataset: X={X_full.shape}, y={y_full.shape}')
print(f'Training samples count: {len(train_idx)}')

## T2 Setup: Reduced 4-Class Subset

**Classes used:** AnnualCrop (0), Forest (1), Industrial (4), SeaLake (9)

**Preprocessing:** 64×64 RGB → 16×16 Grayscale → flatten 256 features, scale to [0,1]

In [ ]:
from PIL import Image

# 4 visually distinct classes for fast CPU training
SUBSET_CLASSES = {0: 0, 1: 1, 4: 2, 9: 3}   # original_label -> new_label
CLASS_NAMES    = ['AnnualCrop', 'Forest', 'Industrial', 'SeaLake']

# Use training indices only
mask = np.isin(y_full[train_idx], list(SUBSET_CLASSES.keys()))
sub_idx = train_idx[mask]

X_raw = X_full[sub_idx]   # (n, 64, 64, 3) uint8
y_raw = y_full[sub_idx]

# Remap labels to 0-3
y_sub = np.array([SUBSET_CLASSES[yi] for yi in y_raw])

# Downsample to 16x16 grayscale + flatten
def preprocess(imgs):
    out = []
    for img in imgs:
        gray = Image.fromarray(img).convert('L').resize((16, 16))
        out.append(np.array(gray, dtype=np.float32).flatten() / 255.0)
    return np.stack(out)   # (n, 256)

X_sub = preprocess(X_raw)

print(f'Subset: {X_sub.shape}, labels: {y_sub.shape}')
print(f'Classes: {CLASS_NAMES}')
print(f'Class distribution: {dict(zip(CLASS_NAMES, np.bincount(y_sub)))}')
print(f'Feature range: [{X_sub.min():.2f}, {X_sub.max():.2f}]')

## T2a: MLP Class (NumPy only — no autograd)

**Architecture:** Input(256) → tanh → Hidden(32) → Softmax → Output(4)

**Biological analogy:**
- Inputs resemble dendritic signals received by a neuron.
- Weights resemble synaptic strengths that scale each signal.
- The weighted sum + bias resembles the membrane potential built up inside the neuron.
- The activation (tanh) resembles the neuron's firing response — compressing the potential into a bounded output.

**Two limits vs real neurons:** real neurons communicate via discrete spikes with timing (rate coding), and backpropagation has no known biological equivalent.

In [ ]:
class MLP:
    """One-hidden-layer MLP in pure NumPy.
    d: input dim, h: hidden units, k: num classes"""

    def __init__(self, d, h, k, seed=0):
        r = np.random.default_rng(seed)
        # He-style init for tanh: scale = sqrt(1/fan_in)
        self.W1 = r.normal(0, np.sqrt(1 / d), (d, h))
        self.b1 = np.zeros(h)
        self.W2 = r.normal(0, np.sqrt(1 / h), (h, k))
        self.b2 = np.zeros(k)

    def forward(self, X):
        """Forward pass. Stores cache for backward."""
        self.X = X                              # cache input
        self.H = np.tanh(X @ self.W1 + self.b1)  # hidden activations
        Z2 = self.H @ self.W2 + self.b2         # logits
        # Numerical stability: subtract row-max before exp
        Z2 = Z2 - Z2.max(axis=1, keepdims=True)
        E = np.exp(Z2)
        self.P = E / E.sum(axis=1, keepdims=True)  # softmax probabilities
        return self.P

    def loss(self, X, y):
        """Mean cross-entropy loss."""
        P = self.forward(X)
        return -np.log(P[np.arange(len(y)), y] + 1e-12).mean()

    def grads(self, X, y):
        """Backpropagation — returns gradient dict."""
        n = len(y)
        P = self.forward(X).copy()
        P[np.arange(n), y] -= 1          # dL/dZ2 = (P - one_hot(y)) / n
        dZ2 = P / n
        dW2 = self.H.T @ dZ2             # chain rule: H^T . dZ2
        db2 = dZ2.sum(axis=0)
        dZ1 = (dZ2 @ self.W2.T) * (1 - self.H ** 2)  # tanh derivative: 1-tanh^2
        dW1 = self.X.T @ dZ1
        db1 = dZ1.sum(axis=0)
        return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

    def step(self, grads, lr):
        """Gradient descent update."""
        for k in grads:
            setattr(self, k, getattr(self, k) - lr * grads[k])


def train(model, X, y, lr=0.1, epochs=200, batch_size=64, seed=0):
    """Mini-batch SGD. Returns loss curve."""
    rng   = np.random.default_rng(seed)
    curve = []
    for ep in range(epochs):
        order = rng.permutation(len(y))
        for i in range(0, len(y), batch_size):
            b = order[i:i + batch_size]
            model.step(model.grads(X[b], y[b]), lr)
        curve.append(model.loss(X, y))
    return curve

print('MLP class defined.')

## T2b: Train & Plot Loss Curve

In [ ]:
LR      = 0.1
EPOCHS  = 200
BATCH   = 64
HIDDEN  = 32

mlp = MLP(d=256, h=HIDDEN, k=4, seed=0)

# Sanity check: initial loss should be close to ln(4) = 1.386 for 4 balanced classes
init_loss = mlp.loss(X_sub, y_sub)
print(f'Initial loss: {init_loss:.4f}  (expected ≈ ln(4) = {np.log(4):.4f})')

curve = train(mlp, X_sub, y_sub, lr=LR, epochs=EPOCHS, batch_size=BATCH, seed=0)

print(f'Final loss  : {curve[-1]:.4f}')
print(f'Settings    : lr={LR}, batch_size={BATCH}, epochs={EPOCHS}, hidden={HIDDEN}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(curve, color='steelblue', linewidth=2)
ax.axhline(np.log(4), color='red', linestyle='--', label=f'Random init baseline ln(4)={np.log(4):.3f}')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax.set_title('T2 NumPy MLP — Training Loss Curve', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/t2_loss_curve.png', dpi=150)
# Also backup to Drive
!cp figures/t2_loss_curve.png /content/drive/MyDrive/ann_cw1/figures/ 2>/dev/null || true
plt.show()
print('Saved: figures/t2_loss_curve.png')

## T2c: Gradient Verification (Central Differences)

Formula: `(L(w+ε) - L(w-ε)) / 2ε` compared to analytic gradient.

**ε = 1e-5**, tolerance: relative error < 1e-6

In [ ]:
def gradcheck(model, X, y, eps=1e-5, per_param=3, seed=1):
    """Central-difference gradient verification.
    Uses tanh (smooth — no ReLU kinks). Checks W1, b1, W2, b2."""
    rng = np.random.default_rng(seed)
    g   = model.grads(X, y)   # analytic gradients
    rows = []

    for name in ['W1', 'b1', 'W2', 'b2']:
        p = getattr(model, name)
        for _ in range(per_param):
            idx = tuple(int(rng.integers(0, s)) for s in p.shape)
            old = p[idx]
            p[idx] = old + eps;  lp = model.loss(X, y)
            p[idx] = old - eps;  lm = model.loss(X, y)
            p[idx] = old
            numeric  = (lp - lm) / (2 * eps)
            analytic = g[name][idx]
            abs_err  = abs(analytic - numeric)
            rel_err  = abs_err / max(abs(analytic) + abs(numeric), 1e-12)
            rows.append((name, idx, analytic, numeric, abs_err, rel_err))

    print(f'{"Param":<6} {"Index":<12} {"Analytic":>12} {"Numeric":>12} {"Abs Err":>10} {"Rel Err":>10}  OK?')
    print('-' * 70)
    for name, idx, ana, num, ab, rel in rows:
        ok = '✅' if rel < 1e-4 else '❌'
        print(f'{name:<6} {str(idx):<12} {ana:>12.4e} {num:>12.4e} {ab:>10.2e} {rel:>10.2e}  {ok}')
    return rows

# Use a small batch for speed
X_check = X_sub[:8]
y_check = y_sub[:8]
mlp_check = MLP(d=256, h=32, k=4, seed=42)   # fresh model for clean grads

rows = gradcheck(mlp_check, X_check, y_check, eps=1e-5, per_param=3)

## T2d: Numerical Diagnosis — Overfit a Tiny Batch

**Protocol:** train on exactly 10 images. If the loss approaches zero, the forward/backward code is plausibly correct.

In [ ]:
TINY_N   = 10
TINY_LR  = 0.5
MAX_EP   = 2000
STOP_THR = 1e-3

mlp_tiny = MLP(d=256, h=32, k=4, seed=0)
X_tiny   = X_sub[:TINY_N]
y_tiny   = y_sub[:TINY_N]

tiny_curve = []
stopped_at = MAX_EP
for ep in range(MAX_EP):
    mlp_tiny.step(mlp_tiny.grads(X_tiny, y_tiny), TINY_LR)
    l = mlp_tiny.loss(X_tiny, y_tiny)
    tiny_curve.append(l)
    if l < STOP_THR:
        stopped_at = ep + 1
        break
print(f'Stopped at epoch : {stopped_at}')
print(f'Final tiny loss  : {tiny_curve[-1]:.6f}')
print('Interpretation   : loss near zero -> forward + backward code is correct.')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(tiny_curve, color='darkorange', linewidth=2)
ax.axhline(STOP_THR, color='red', linestyle='--', label=f'Stop threshold={STOP_THR}')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax.set_title(f'T2d Overfit on {TINY_N} images — Numerical Diagnosis', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/t2_overfit_diagnosis.png', dpi=150)
# Also backup to Drive
!cp figures/t2_overfit_diagnosis.png /content/drive/MyDrive/ann_cw1/figures/ 2>/dev/null || true
plt.show()
print('Saved: figures/t2_overfit_diagnosis.png')

---
## ✅ Notebook 03 Complete
**Next:** Run `04_baseline_hog_lr.ipynb` for the T3a HOG + Logistic Regression baseline.